# 2.3 — Longitudinal Normative Modeling: Prediction & Delta-Z

Applies the pre-trained BLR normative model to all test subjects across all timepoints, computes longitudinal delta-Z scores, and evaluates accuracy metrics.

**Inputs:**
- `healthy_test.csv` — healthy test subjects (output of `01.3_train_test_split.ipynb`)
- `clinical_test.csv` — clinical test subjects (output of `01.3_train_test_split.ipynb`)
- Trained BLR models from `02.1_normative_model_blr.ipynb` (one `Models/` folder per ROI)

**Outputs:** Z-scores, delta-Z (all visit pairs), global accuracy metrics  
**Environment:** `normmodel310` (PCNToolkit ≥ 0.29)

In [1]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Directory that contains the per-ROI model folders (output of 02.1_normative_model_blr.ipynb)
BASE_DIR    = 'outputs/normative_model'

# Directory containing healthy_test.csv and clinical_test.csv (output of 01.3_train_test_split.ipynb)
DATA_DIR    = 'data/splits'

# All prediction outputs are written here
RESULTS_DIR = 'outputs/longitudinal'

# B-spline age range — must match training (02.1_normative_model_blr.ipynb)
AGE_XMIN, AGE_XMAX, AGE_NKNOTS = 7, 18, 3

In [2]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pcntoolkit.normative import predict, evaluate
from pcntoolkit.util.utils import create_bspline_basis

os.makedirs(RESULTS_DIR, exist_ok=True)

## Step 1 — Load & merge test data

In [3]:
SITE_MAPPING = {
    'HASH4d1ed7b1_9': 1,  'HASH3935c89e_16': 2,  'HASHd7cb4c6d_10': 3,
    'HASHe3ce02d3_10': 4, 'HASH7911780b_15': 5,  'HASH3ce956bd_18': 6,
    'HASHa3e45734_18': 7, 'HASH4036a433_21': 8,  'HASHdb2589d4_17': 9,
    'HASH1314a204_2': 10, 'HASHfeb7e81a_4': 11,  'HASHc3bf3d9c_13': 12,
    'HASH69f406fa_13': 13,'HASH96a0c182_6': 14,  'HASHe4f6957a_12': 15,
    'HASH4b0b8b05_4': 16, 'HASH11ad4ed5_14': 17, 'HASH7f91147d_14': 18,
    'HASH65b39280_7': 19, 'HASH311170b9_5': 20,  'HASHe76e6d72_21': 21,
    'HASHb640a1b8_21': 22,'HASH03db707f_11': 23, 'HASHd422be27_20': 24,
    'HASHc9398971_20': 25,'HASH5b0cf1bb_3': 26,  'HASH5b2fcf80_8': 27,
    'HASH5ac2b20b_19': 28,'HASHc1b3365b_13': 29, 'HASH6b4422a7_1': 30,
}

VALID_VISITS = ['ses-00A', 'ses-02A', 'ses-04A', 'ses-06A']
DROP_COLS    = [
    'mr_y_smri__vol__dsk__lh_sum',
    'mr_y_smri__vol__dsk__rh_sum',
    'mr_y_smri__vol__dsk_sum',
]

df_c = pd.read_csv(os.path.join(DATA_DIR, 'healthy_test.csv'))
df_p = pd.read_csv(os.path.join(DATA_DIR, 'clinical_test.csv'))
df_c['group'] = 'Control'
df_p['group'] = 'Patient'

all_data = pd.concat([df_c, df_p], axis=0).reset_index(drop=True)
all_data  = all_data.drop(columns=[c for c in DROP_COLS if c in all_data.columns])
all_data  = all_data[all_data['session_id'].isin(VALID_VISITS)].copy()

all_data['ID-wave'] = all_data['participant_id'] + '_' + all_data['session_id']
all_data = all_data.set_index('ID-wave')
all_data['site_mri'] = all_data['site_mri'].map(SITE_MAPPING).astype('Int64')
all_data = all_data.dropna(subset=['site_mri', 'ab_g_dyn__visit_age'])

roi_ids = all_data.columns[all_data.columns.str.contains('mr_y_smri__')].tolist()
print(f'Total rows: {len(all_data)} | ROIs: {len(roi_ids)}')
print(all_data.groupby('group')['group'].count().to_string())

FileNotFoundError: [Errno 2] No such file or directory: 'data/splits/healthy_test.csv'

## Step 2 — Prepare B-spline covariates

In [ ]:
cov_cols = ['ab_g_dyn__visit_age', 'ab_g_stc__cohort_sex', 'site_mri']
X_df     = pd.get_dummies(all_data[cov_cols], columns=['site_mri'], dtype=int)

B        = create_bspline_basis(AGE_XMIN, AGE_XMAX, nknots=AGE_NKNOTS)
X_arr    = X_df.values
splines  = np.vstack([B(age) for age in X_arr[:, 0]])
X_pred   = np.hstack([X_arr, np.ones((X_arr.shape[0], 1)), splines])

cov_file = os.path.join(RESULTS_DIR, 'cov_predict.txt')
np.savetxt(cov_file, X_pred)

# Save per-ROI response files
for roi in roi_ids:
    all_data[roi].to_csv(os.path.join(RESULTS_DIR, f'resp_{roi}.txt'), header=False, index=False)

print(f'Covariate matrix: {X_pred.shape}  saved to {cov_file}')

## Step 3 — Prediction loop

In [ ]:
df_yhat = pd.DataFrame(index=all_data.index, columns=roi_ids)
df_ys2  = pd.DataFrame(index=all_data.index, columns=roi_ids)
df_Z    = pd.DataFrame(index=all_data.index, columns=roi_ids)
df_y    = all_data[roi_ids].copy()

root_dir = os.getcwd()

for roi in roi_ids:
    roi_model_dir = os.path.join(BASE_DIR, roi)
    roi_out_dir   = os.path.join(RESULTS_DIR, roi)
    model_folder  = os.path.join(roi_model_dir, 'Models')
    os.makedirs(roi_out_dir, exist_ok=True)

    # Detect model filename (NM_0_0 vs Model_1) and normalise
    source_model = os.path.join(model_folder, 'NM_0_0_estimate.pkl')
    target_model = os.path.join(model_folder, 'Model_1_estimate.pkl')
    if os.path.exists(source_model) and not os.path.exists(target_model):
        shutil.copy(source_model, target_model)

    if os.path.exists(target_model):
        suffix = 'estimate'
    elif os.path.exists(os.path.join(model_folder, 'Model_1.pkl')):
        suffix = ''
    else:
        print(f'  SKIP {roi}: no model found')
        continue

    os.chdir(roi_out_dir)  # PCNToolkit writes output relative to cwd
    try:
        predict(
            covfile=cov_file,
            respfile=os.path.join(RESULTS_DIR, f'resp_{roi}.txt'),
            alg='blr',
            model_path=model_folder,
            output_path=roi_out_dir,
            inputsuffix=suffix,
            outputsuffix='pred',
        )

        yhat_files = glob.glob('*yhat*pred.txt')
        ys2_files  = glob.glob('*ys2*pred.txt')
        Z_files    = glob.glob('*Z*pred.txt')

        # Fallback: files written to parent directory
        if not yhat_files:
            for f in glob.glob(os.path.join(RESULTS_DIR, '*yhat*pred.txt')):
                shutil.move(f, roi_out_dir)
            for f in glob.glob(os.path.join(RESULTS_DIR, '*ys2*pred.txt')):
                shutil.move(f, roi_out_dir)
            for f in glob.glob(os.path.join(RESULTS_DIR, '*Z*pred.txt')):
                shutil.move(f, roi_out_dir)
            yhat_files = glob.glob('*yhat*pred.txt')
            ys2_files  = glob.glob('*ys2*pred.txt')
            Z_files    = glob.glob('*Z*pred.txt')

        if not yhat_files:
            print(f'  MISSING output: {roi}')
            continue

        df_yhat[roi] = np.genfromtxt(yhat_files[0])
        df_ys2[roi]  = np.genfromtxt(ys2_files[0])
        df_Z[roi]    = np.genfromtxt(Z_files[0])

    except Exception as e:
        print(f'  ERROR {roi}: {e}')

os.chdir(root_dir)
print('Prediction loop complete.')

## Step 4 — Save master prediction files

In [ ]:
if df_Z.dropna(how='all').empty:
    raise RuntimeError('No Z-scores collected — check model paths and prediction loop output above.')

df_y.to_csv(   os.path.join(RESULTS_DIR, 'Y_all_timepoints.csv'))
df_yhat.to_csv(os.path.join(RESULTS_DIR, 'Yhat_all_timepoints.csv'))
df_ys2.to_csv( os.path.join(RESULTS_DIR, 'Ys2_all_timepoints.csv'))
df_Z.to_csv(   os.path.join(RESULTS_DIR, 'Z_all_timepoints.csv'))

print('Saved Y, Yhat, Ys2, Z for all timepoints.')

## Step 5 — Longitudinal delta-Z

In [ ]:
def get_visit_Z(df_z, visit_id):
    subset = df_z[df_z.index.str.contains(visit_id)].copy()
    subset.index = subset.index.str.split('_ses').str[0]  # strip session suffix
    return subset[~subset.index.duplicated(keep='first')]

visits  = ['ses-00A', 'ses-02A', 'ses-04A', 'ses-06A']
v_labels = ['V1', 'V2', 'V3', 'V4']

subject_group_map = (
    all_data.reset_index()
    .drop_duplicates(subset='participant_id')
    .set_index('participant_id')['group']
    .to_dict()
)

for i in range(len(visits)):
    for j in range(i + 1, len(visits)):
        label   = f'{v_labels[i]}_{v_labels[j]}'
        Z_start = get_visit_Z(df_Z, visits[i])
        Z_end   = get_visit_Z(df_Z, visits[j])

        common = Z_start.index.intersection(Z_end.index)
        if len(common) == 0:
            continue

        delta_raw = Z_end.loc[common] - Z_start.loc[common]
        groups    = delta_raw.index.map(subject_group_map)
        ctrl_mask = (groups == 'Control')

        if ctrl_mask.sum() < 2:
            print(f'  Skipping {label}: insufficient controls ({ctrl_mask.sum()})')
            continue

        # Standardise by control SD of change
        ctrl_sd = delta_raw.loc[ctrl_mask].std(axis=0).replace(0, 1)
        delta_Z = delta_raw.div(ctrl_sd, axis=1)
        delta_Z.insert(0, 'group', groups)
        delta_Z.index.name = 'ID'

        save_path = os.path.join(RESULTS_DIR, f'DeltaZ_{label}.csv')
        delta_Z.to_csv(save_path)
        print(f'  Saved {label}: {len(delta_Z)} subjects')

## Step 6 — Accuracy metrics on control group

In [ ]:
# Evaluate on all timepoints from controls (healthy held-out data)
ctrl_idx  = all_data[all_data['group'] == 'Control'].index
Y_eval    = df_y.loc[ctrl_idx]
Yhat_eval = df_yhat.loc[ctrl_idx]
S2_eval   = df_ys2.loc[ctrl_idx]

blr_globmetrics = pd.DataFrame(columns=['ROI','site','MSLL','EXPV','SMSE','RMSE','Rho','pRho'])

for roi in roi_ids:
    mask  = ~Y_eval[roi].isna() & ~Yhat_eval[roi].isna()
    y_col    = Y_eval.loc[mask, roi].values
    yhat_col = Yhat_eval.loc[mask, roi].values
    s2_col   = S2_eval.loc[mask, roi].values

    if len(y_col) < 2:
        continue

    metrics = evaluate(
        y_col.reshape(-1, 1),
        yhat_col.reshape(-1, 1),
        s2_col.reshape(-1, 1),
        np.array([y_col.mean()]),
        np.array([y_col.std()]),
    )
    blr_globmetrics.loc[len(blr_globmetrics)] = [
        roi, 'global',
        metrics['MSLL'][0], metrics['EXPV'][0],
        metrics['SMSE'][0], metrics['RMSE'][0],
        metrics['Rho'][0],  metrics['pRho'][0],
    ]

metrics_path = os.path.join(RESULTS_DIR, 'Global_Accuracy_Metrics_longitudinal.csv')
blr_globmetrics.to_csv(metrics_path, index=False)

print(f'Saved: {metrics_path}')
print(f'Mean EXPV : {blr_globmetrics["EXPV"].mean():.4f}')
print(f'Mean Rho  : {blr_globmetrics["Rho"].mean():.4f}')
print('Pipeline complete.')